In [1]:
import sys 
from pathlib import Path 
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import PROCESSED_DATA_DIR, BERT_DIR
from src import model_utils

In [2]:
reviews_df = pd.read_parquet(PROCESSED_DATA_DIR / "amazon_reviews_clean.parquet")

In [3]:
print(reviews_df.columns)

Index(['country', 'review_count', 'review_date', 'rating', 'review_title',
       'review_text', 'date_of_experience'],
      dtype='str')


In [4]:
reviews_df["full_text"] = (reviews_df["review_title"] + " " + reviews_df["review_text"]).str.strip()
print(reviews_df.columns)

Index(['country', 'review_count', 'review_date', 'rating', 'review_title',
       'review_text', 'date_of_experience', 'full_text'],
      dtype='str')


In [5]:
reviews_df = reviews_df[["rating", "full_text"]]
print(reviews_df.columns)

Index(['rating', 'full_text'], dtype='str')


In [6]:
reviews_df = model_utils.drop_neutral_ratings(reviews_df)
print(reviews_df["rating"].unique())

[1 5 2 4]


In [7]:
reviews_df["rating"] = reviews_df["rating"].apply(model_utils.classify_rating)
print(reviews_df["rating"].unique())

[0 1]


In [8]:
# Drop duplicates
reviews_df = reviews_df.drop_duplicates().reset_index(drop=True)

In [9]:
# Save dataset to disk
reviews_df.to_parquet(BERT_DIR / "amazon_reviews_bert.parquet", index=False)